In [ ]:
dnaFastaPerOG = mclapply(list.files("/workdir/sh2246/p_phyloGWAS/output/CDSMSAPerOG_gs",pattern = "*.gs.l0.fa$",
                                   full.names = T),function(x) readDNAStringSet(x),mc.cores = 20)

names(dnaFastaPerOG) = gsub(".gs.l0.fa","",list.files("/workdir/sh2246/p_phyloGWAS/output/CDSMSAPerOG_gs",pattern = "*.gs.l0.fa$"))

gc.og = mclapply(dnaFastaPerOG,function(x){
    tmp = letterFrequency(x,letters = c("A","T","C","G"))
    rownames(tmp) = names(x)
    tmp = apply(tmp,2,function(x) tapply(x,limma::strsplit2(rownames(tmp),":")[,2],sum))
    tmp.gc = rowSums(tmp[,3:4])/rowSums(tmp)
    return(tmp.gc[commonID])
},mc.cores = 30)

gc.og = simplify2array(gc.og)

In [ ]:
fwrite(gc.og,"/workdir/sh2246/p_phyloGWAS/output/GCperOG_20250729.txt",sep = "\t",row.names = T,quote = F)

In [ ]:
gc.og = fread("/workdir/sh2246/p_phyloGWAS/output/GCperOG_20250729.txt",data.table = F,header = T)
rownames(gc.og) = gc.og[,1]
gc.og = as.matrix(gc.og[,-1])

In [ ]:
gc.og.avg = apply(gc.og,2,mean,na.rm = T)
gc.og.sd = apply(gc.og,2,sd,na.rm = T)

In [ ]:
options(repr.plot.width=10, repr.plot.height=8)
par(mfrow = c(1,2))
hist(gc.og.avg,breaks = seq(0.2,1,0.02),main = "avg. GC% per OG",xlab = "")
hist(gc.og.sd,breaks = seq(0,.3,0.01),main = "S.D. of GC% per OG",xlab = "")

In [ ]:
LSD::heatscatter(t(gc.og)[,1],t(gc.og)[,279],xlab = "wheat (Jagger)",ylab = "maize (B73)",main = "wheat vs. maize")
cor(t(gc.og)[,c(1,279)],use = "pairwise.complete.obs")

In [ ]:
LSD::heatscatter(t(gc.og)[,1],t(gc.og)[,120],xlab = "wheat (Jagger)",ylab = "Rice (Nipponbare)",main = "wheat vs. rice")
cor(t(gc.og)[,c(1,120)],use = "pairwise.complete.obs")

In [ ]:
gc.og.filled = gc.og
gc.og.filled[is.na(gc.og.filled)] = mean(gc.og,na.rm = T)

In [ ]:
### 4d

In [ ]:
get_4fold_sites <- function(ref_seq) {
  codons <- substring(ref_seq, seq(1, nchar(ref_seq) - 2, by = 3), seq(3, nchar(ref_seq), by = 3))

  # Define all 4-fold degenerate codons
  codon_table_4fold <- c(
    "GCT","GCC","GCA","GCG",  # Ala
    "CGT","CGC","CGA","CGG",  # Arg
    "GGT","GGC","GGA","GGG",  # Gly
    "CCT","CCC","CCA","CCG",  # Pro
    "ACT","ACC","ACA","ACG",  # Thr
    "GTT","GTC","GTA","GTG",  # Val
    "TCT","TCC","TCA","TCG",  # Ser
    "CTT","CTC","CTA","CTG"   # Leu
  )

  is_4fold <- codons %in% codon_table_4fold
  third_base_indices <- which(is_4fold) * 3
  return(third_base_indices)
}


In [ ]:
dnaFastaPerOG4d = mclapply(dnaFastaPerOG,function(x) {
    gc()
    ref_seq = x[grep("ASM1935983v1",names(x))[1]]
    msa_mat = as.matrix(x)
    msa_mat_4d = msa_mat[,get_4fold_sites(ref_seq)]
    if(is.null(msa_mat_4d)) return(NULL)
    else{
        msa_seqs <- apply(msa_mat_4d, 1, paste0, collapse = "")

    # Convert back to DNAStringSet
    msa_dna <- DNAStringSet(msa_seqs)
    names(msa_dna) <- rownames(msa_mat)
    return(msa_dna)
    }
    
},mc.cores = 30)

dnaFastaPerOG4d = dnaFastaPerOG4d[sapply(dnaFastaPerOG4d,length)!=1]

gc.og.4d = mclapply(dnaFastaPerOG4d,function(x){
    tmp = letterFrequency(x,letters = c("A","T","C","G"))
    rownames(tmp) = names(x)
    tmp = apply(tmp,2,function(x) tapply(x,limma::strsplit2(rownames(tmp),":")[,2],sum))
    tmp = as.data.frame(tmp)
    tmp.gc = rowSums(tmp[,3:4])/rowSums(tmp)
    return(tmp.gc[commonID])
},mc.cores = 30)

gc.og.4d= simplify2array(gc.og.4d)

In [ ]:
dna.comp.og.4d = mclapply(dnaFastaPerOG4d,function(x){
    tmp = letterFrequency(x,letters = c("A","T","C","G"))
    rownames(tmp) = names(x)
    tmp = apply(tmp,2,function(x) tapply(x,limma::strsplit2(rownames(tmp),":")[,2],sum))
    tmp = as.data.frame(tmp)
    return(tmp[commonID,])
},mc.cores = 30)



In [ ]:
options(repr.plot.width=8, repr.plot.height=8)
hist(gc.og.4d[279,],breaks = seq(0,1,0.02),main = "GC% at 4-fold degenerate site per OG in Maize (B73)",xlab = "")

In [ ]:
LSD::heatscatter(t(gc.og)[colnames(gc.og.4d),279],t(gc.og.4d)[,279],
                 xlab = "all coding sites",ylab = "4-fold degenerate site",main = "maize (B73)")
abline(a=0,b=1)

In [ ]:
gc.og.4d.avg = apply(gc.og.4d,2,mean,na.rm = T)
gc.og.4d.sd = apply(gc.og.4d,2,sd,na.rm = T)

In [ ]:
hist(gc.og.4d.avg,breaks = seq(0,1,0.02),main = "avg. GC% at 4d site per OG",xlab = "")

In [ ]:
LSD::heatscatter(t(gc.og.4d)[,279],t(gc.og.4d)[,120],
                 xlab = "Maize (B73)",ylab = "Rice (Nipponbare)",main = "maize vs. rice")
abline(a=0,b=1)
cor(t(gc.og.4d)[,c(279,120)],use = "pairwise.complete.obs")

In [ ]:
dnaFastaPerOG4d50 = mclapply(dnaFastaPerOG,function(x) {
    gc()
    ref_seq = x[grep("ASM1935983v1",names(x))[1]]
    msa_mat = as.matrix(x)
    Idx4d = get_4fold_sites(ref_seq)
    if (width(ref_seq)>150) Idx4d = Idx4d[Idx4d<150]
    if(length(Idx4d)!=0) {
        msa_mat_4d = msa_mat[,Idx4d]
        msa_seqs <- apply(msa_mat_4d, 1, paste0, collapse = "")

    # Convert back to DNAStringSet
    msa_dna <- DNAStringSet(msa_seqs)
    names(msa_dna) <- rownames(msa_mat)
    return(msa_dna)
    } else return(NULL)
    
},mc.cores = 30)

dnaFastaPerOG4d50 = dnaFastaPerOG4d50[!sapply(dnaFastaPerOG4d50,is.null)]

gc.og.4d.50 = mclapply(dnaFastaPerOG4d50,function(x){
    tmp = letterFrequency(x,letters = c("A","T","C","G"))
    rownames(tmp) = names(x)
    tmp = apply(tmp,2,function(x) tapply(x,limma::strsplit2(rownames(tmp),":")[,2],sum))
    tmp = as.data.frame(tmp)
    tmp.gc = rowSums(tmp[,3:4])/rowSums(tmp)
    return(tmp.gc[commonID])
},mc.cores = 30)

gc.og.4d.50= simplify2array(gc.og.4d.50)

In [ ]:
dnaFastaPerOG4d50up = mclapply(dnaFastaPerOG,function(x) {
    gc()
    ref_seq = x[grep("ASM1935983v1",names(x))[1]]
    msa_mat = as.matrix(x)
    Idx4d = get_4fold_sites(ref_seq)
    if(width(ref_seq)>150) Idx4d = Idx4d[Idx4d>150] else{Idx4d=NULL}
    msa_mat_4d = msa_mat[,Idx4d]
    if(length(Idx4d)!=0){
        msa_seqs <- apply(msa_mat_4d, 1, paste0, collapse = "")
        msa_dna <- DNAStringSet(msa_seqs)
        names(msa_dna) <- rownames(msa_mat)
        return(msa_dna)
    } else return(NULL)
},mc.cores = 30)

dnaFastaPerOG4d50up = dnaFastaPerOG4d50up[!sapply(dnaFastaPerOG4d50up,is.null)]

gc.og.4d.50up = mclapply(dnaFastaPerOG4d50up,function(x){
    tmp = letterFrequency(x,letters = c("A","T","C","G"))
    rownames(tmp) = names(x)
    tmp = apply(tmp,2,function(x) tapply(x,limma::strsplit2(rownames(tmp),":")[,2],sum))
    tmp = as.data.frame(tmp)
    tmp.gc = rowSums(tmp[,3:4])/rowSums(tmp)
    return(tmp.gc[commonID])
},mc.cores = 30)

gc.og.4d.50up= simplify2array(gc.og.4d.50up)

In [ ]:
gc.og.4d.50.avg = apply(gc.og.4d.50,2,mean,na.rm = T)

In [ ]:
LSD::heatscatter(log10(OGLength[colnames(gc.og.4d)]),t(gc.og.4d)[,279],
                 xlab = expression(log[10](length)),ylab = "GC4d",main = "")

In [ ]:
LSD::heatscatter(log10(OGLength[colnames(gc.og.4d)]),t(gc.og.4d.50)[,279],
                 xlab = expression(log[10](length)),ylab = "GC4d50",main = "")

In [ ]:
testRes_envPC1_v2 = read.table("/workdir/sh2246/p_phyloGWAS/output/ASREML_res_envPC1_20250811.txt",header =T)
sigGeneEnvPC1 = na.omit(testRes_envPC1_v2$OG[testRes_envPC1_v2$p<.001])
testRes_envPC2_v2 = read.table("/workdir/sh2246/p_phyloGWAS/output/ASREML_res_envPC2_20250811.txt",header =T)
sigGeneEnvPC2 = na.omit(testRes_envPC2_v2$OG[testRes_envPC2_v2$p<.001])

In [ ]:
plot(density(gc.og.4d.50.avg[setdiff(testRes_envPC1_v2$OG,union(sigGeneEnvPC1,sigGeneEnvPC2))],na.rm = T),
     col = alpha("grey",1),lwd = 2,main = "",xlab = "GC% at 4d site of the first 50 codons")
lines(density(gc.og.4d.50.avg[sigGeneEnvPC1],na.rm = T),
      col = alpha("royalblue",1),lwd = 2)
lines(density(gc.og.4d.50.avg[sigGeneEnvPC2],na.rm = T),
      col = alpha("brown",1),lwd = 2)
legend("topleft",lty = 1, lwd = 2,col = c("royalblue","brown","grey"),
       legend = c("envPC1 (232)","envPC2 (122)","n.s. (17681)"))